In [ ]:
%matplotlib inline
from matplotlib import rcParams

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Introduction to clustering

Clustering is a machine learning technique used to group unlabeled datapoints in groups called *clusters*. The grouping is done such that similar items are together in the same cluster and items in different clusters are dissimilar from each-other. Typically similarity is defined using a distance measure (Euclidean, cosine, etc). Common practical applications for clustering include: market segmentation, social network analysis, search result grouping, anomaly detection etc.


Two clustering methods will be explored in this notebook:

- K-means.
- Hierarchical clustering.

More methods can be found [here](https://scikit-learn.org/stable/modules/classes.html?highlight=cluster#module-sklearn.cluster).


#### Resources: 
* Clustering & K-means (fr): https://www.youtube.com/watch?v=FTtzd31IAOw (up to 11min)
* Clustering & K-means (en): https://www.youtube.com/watch?v=QXOkPvFM6NU  (up to 7min46)
* K-means algorithm : https://stanford.edu/~cpiech/cs221/handouts/kmeans.html

* Hierarchical clustering example with scikit-learn : https://stackabuse.com/hierarchical-clustering-with-python-and-scikit-learn/ (example on how to do hierarchical clustering with scikit-learn)
* Scikit doc for hierarchical clustering : https://scikit-learn.org/stable/modules/clustering.html#hierarchical-clustering

We will not use the elbow method presented in the videos as it is flawed. Instead you will learn the silhouette plots and scores to decide the number of clusters.

### Subject comprehension 

To check your understanding, before moving further, answer the questions in your own words in the space below.

1. What is clustering?
 It is an unsupervised learning technique that groups similar data points together
2. What are the main steps in the standard process of clustering?
Clean the data set, choose a clustering model, choose or find the number of clusters, fit and evaluate the results
3. What is the Silhouette coefficient?
A metric that tells us how well a point fits within a cluster compare to others. From -1 to 1
4. What are the two types of evaluation for a clustering algorithm and what is the difference between them?


*Answer here*

### Data acquisition

Load the data about the countries statistics ( *Country-data.csv* ) file into a pandas dataframe. Make the necessary imports.

In [ ]:
# code here
df = pd.read_csv("../data/Country-data.csv")

Print a summary of the data as well as the first 5 rows.

In [ ]:
# code here
df.info()

### Preprocessing

To facilitate the computation of distances, keep only the numerical features in the dataset.

- Remove all non-numerical features, as well as the features related to the date.
- Make sure there are no missing values - if they exist, remove them.
- Normalize the new dataset (transform the data for multiple features to the same scale).
- Save the result in a dataframe *df_norm*

In [ ]:
# code here
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

df_clean = df.drop(columns = 'country')
model = StandardScaler()
df_norm = pd.DataFrame(model.fit_transform(df_clean),columns=df_clean.columns,index=df_clean.index)

In [ ]:
df_norm

Plot histograms of features

In [ ]:
# code here
rcParams['figure.figsize'] = 12,7
plots = df_norm.hist()
plt.show()

## KMeans


Let's warm up with the well known algorithm named KMeans!

Centroid-based clustering organizes the data into non-hierarchical clusters, contrarily to hierarchical clustering. KMeans is the most widely-used centroid-based clustering algorithm. Centroid-based algorithms are efficient but sensitive to initial conditions and outliers. 


To better understand the way the algorithm works, you can look at the following resources :
* Clustering & K-means (fr): https://www.youtube.com/watch?v=FTtzd31IAOw (up to 11min)
* Clustering & K-means (en): https://www.youtube.com/watch?v=QXOkPvFM6NU  (up to 7min46)
* https://stanford.edu/~cpiech/cs221/handouts/kmeans.html


Then answer the below questions (in your own words):
1. What is a centroid?
   The center of a cluster, representing its average position
2. How can you compute a centroid?
    Calculaing the mean of each feaure across all data poi
3. Are you supposed to know in advance the number of clusters in order to use the KMeans algorithm?
   Yes, you must specify it

*Answer here*

#### Make the necessary imports from scikit learn in order to perform KMeans clustering

In [ ]:
# code here
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


#### Exploring KMeans with sklearn

The goal of this step is to become familiar with the sklearn clustering package and the silhouette analysis concepts.

Start by peforming a clustering of the data in *df_norm* into 5 clusters.

In [ ]:
# code here
kmeans = KMeans(n_clusters=5, random_state=0, n_init=10)

kmeans.fit(df_norm)

Print the centroids obtained

In [ ]:
# code here
print(kmeans.cluster_centers_)

Print the cluster labels assigned to each point

In [ ]:
# code here
print(kmeans.labels_)

### **Visualization**

Let's try to visualize the created clusters. As it is easier to visualize 2-dimensional data, first run PCA on the data. Then create a scatterplot to visualize the clustered points, each coloured differently depending on the cluster they belong to.

In [ ]:
# code here
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_norm)
df_pca = pd.DataFrame(df_pca, columns=['PC1', 'PC2'])
df_pca['cluster'] = kmeans.labels_
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='cluster', palette='viridis')

plt.title('KMeans clustering visualization')
plt.show()

In [ ]:
pca = PCA(n_components=3)
df_pca = pca.fit_transform(df_norm)

df_pca = pd.DataFrame(df_pca, columns=['PC1', 'PC2', 'PC3'])
df_pca['cluster'] = kmeans.labels_
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    df_pca['PC1'],
    df_pca['PC2'],
    df_pca['PC3'],
    c=df_pca['cluster'],
    cmap='viridis'
)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')

plt.title('KMeans clustering in 3D')
plt.show()

In [ ]:
results = pd.DataFrame({
    'country': df['country'],
    'cluster': kmeans.labels_
})

print(results.sort_values('cluster'))

In [ ]:
for cluster in sorted(results['cluster'].unique()):
    countries = results[results['cluster'] == cluster]['country'].tolist()
    print(f"Cluster {cluster}: {countries}")

### Evaluation

**Inertia** can be a measure of how internally coherent clusters are. It represents the within-cluster sum-of-squares.

Read more about it here:
https://scikit-learn.org/stable/modules/clustering.html#k-means 

Print the inertia obtained after the clustering.

In [ ]:
# code here
print(kmeans.inertia_)

**The silhouette score**

Make the necessary imports from scikit learn and compute the silhoutte score.

Read what this score represents and how it is computed here: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html and here https://scikit-learn.org/stable/modules/clustering.html#silhouette-coefficient

In [ ]:
# code here
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = kmeans.fit_predict(df_norm)

    score = silhouette_score(df_norm, labels)
    silhouette_scores.append(score)

    print(f"K = {k}, Silhouette Score = {score:.3f}")

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for different K')
plt.show()

You can read more about other clustering evaluation methods here: https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation

#### Finding the number of clusters

In the following article you will find [methods](https://en.wikipedia.org/wiki/Determining_the_number_of_clusters_in_a_data_set) for determining the optimal number of clusters.

The first step for using this algorithm is to use the silhouette analysis technique for obtaining the number of clusters.

(another method you can try is the [elbow method](https://en.wikipedia.org/wiki/Elbow_method_(clustering)) but this is not very reliable)

- Use the silhouette analysis to find an optimal number of clusters

<u><i>Hint:</u></i> Feel free to use the `n_jobs` hyperparameter to speed up your computation

In [ ]:
# code here
# done upper

#### Preprocessing and model testing

Try different preprocessing methods of the data (StandardScaler, MinMaxScaler, ...) (and if you want, different clustering models (KMeans, SpectralClustering,...), and plot again PCA and TSNE to pick the combination that gives a better silhouette score, plot and separation of the clusters on the plots :

In [ ]:
# code here
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
## ELEGIR PREPROCESSING
model = RobustScaler()
# model = MinMaxScaler()
# model = StandardScaler()

##Transformamos los datos
df_norm2 = pd.DataFrame(model.fit_transform(df_clean),columns=df_clean.columns,index=df_clean.index)

#ELEGIR MODELO CLUSTERING
model_clus = KMeans(n_clusters=2, n_init=10, random_state=0)
# model_clus = SpectralClustering(n_clusters = 2, n_init=10, random_state = 0)

In [ ]:
from sklearn.cluster import SpectralClustering

silhouette_scores = []
range_k = list(range(2,11))
for k in range_k:
    model_clus.n_clusters = k #Cambio el numero de clusters a k
    labels = model_clus.fit_predict(df_norm2)

    score = silhouette_score(df_norm2, labels)
    silhouette_scores.append(score)
max_score_sil = np.argmax(silhouette_scores)

print(max(silhouette_scores))
range_k[max_score_sil]

#El maximo silhoutte score se da con RobusScaler y KMeans

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(range_k, silhouette_scores, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for different K')
plt.show()

In [ ]:
# model_clus = KMeans(n_clusters=range_k[max_score_sil], random_state=0, n_init=10)
# model_clus = SpectralClustering(n_clusters = range_k[max_score_sil], n_init=10, random_state = 0)
model_clus.n_clusters = range_k[max_score_sil]
model_clus.fit(df_norm2)

In [ ]:
# code here
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_norm2)
df_pca = pd.DataFrame(df_pca, columns=['PC1', 'PC2'])
df_pca['cluster'] = model_clus.labels_
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='cluster', palette='viridis')

plt.title('KMeans clustering visualization')
plt.show()

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
df_tsne = tsne.fit_transform(df_norm2)

df_tsne = pd.DataFrame(df_tsne, columns=['Dim1', 'Dim2'])
df_tsne['cluster'] = model_clus.labels_

sns.scatterplot(data=df_tsne, x='Dim1', y='Dim2', hue='cluster', palette='viridis')

plt.title('KMeans clustering visualization with t-SNE')
plt.show()

In [ ]:
results = pd.DataFrame({
    'country': df['country'],
    'cluster': model_clus.labels_
})
for cluster in sorted(results['cluster'].unique()):
    countries = results[results['cluster'] == cluster]['country'].tolist()
    print(f"Cluster {cluster}: {countries}")

In [ ]:
df_results = df_clean.copy()
df_results['cluster'] = labels

cluster_means = df_results.groupby('cluster').mean()
global_means = df_clean.mean()

between_variance = (
    df_results.groupby('cluster').size().values[:, None]
    * (cluster_means - global_means) ** 2
).sum(axis=0)

total_variance = ((df_clean - global_means) ** 2).sum()

feature_importance = (between_variance / total_variance).sort_values(ascending=False)

print(feature_importance)

What is the optimal number of clusters that you obtained?

Between 2 and 5. The max silhoutte score is given by RobustScaler + KMeans: ~0.4. Most of the combinations give 2 clusters. Whn clusters = 5 we hace 1 cluter with only 1 data point and 1 cluster with 3

**Comparison between K = 2 and K = 5**
- K = 2: Clusters are mainly differentiated by gdpp (GDP per capita), suggesting a separation based primarily on economic development.
- K = 5: More features contribute to cluster separation, especially child_mort, income, total_fer and life_expec, suggesting a more detailed classification based on economic, demographic and health indicators.

## Hierarchical Clustering

Let's try to apply now a different method on the same data. You can find more explanations on hierarchical clustering in the following links :
 * https://www.youtube.com/watch?v=QXOkPvFM6NU&t=1s (from minute 11:34 onwards)
 * https://stackabuse.com/hierarchical-clustering-with-python-and-scikit-learn/ (example on how to do hierarchical clustering with scikit-learn)
 * https://scikit-learn.org/stable/modules/clustering.html#hierarchical-clustering

Make the necessary imports from scikit learn to perform hierarchical clustering. 

Start by plotting the dendrogram and deciding the number of clusters. Then apply hierarchical clustering.

You might find this useful: https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html for creating the dendrogram.

In [ ]:
# code here
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

In [ ]:
# Calculate the linkage matrix
Z = linkage(df_norm2, method='ward')

# Plot dendrogram
plt.figure(figsize=(15, 7))
dendrogram(Z, no_labels=True)

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Countries')
plt.ylabel('Distance')
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

scores = []

for k in range(2, 11):
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(df_norm2)

    score = silhouette_score(df_norm2, labels)
    scores.append(score)

    print(f"K = {k}, Silhouette Score = {score:.3f}")

In [ ]:
model_h_clus = AgglomerativeClustering(
    n_clusters=2,
    linkage='ward'
)

labels = model_h_clus.fit_predict(df_norm2)

In [ ]:
results = pd.DataFrame({
    'country': df['country'],
    'cluster': labels
})

for cluster in sorted(results['cluster'].unique()):
    countries = results[results['cluster'] == cluster]['country'].tolist()
    print(f"Cluster {cluster}: {countries}")

In [ ]:
print(silhouette_score(df_norm2, labels))

How many clusters did you find?

I choose between 2 and 5 based on what we saw before

Compare the kmeans labels with the labels obtained by the hierarchical clustering.

In [ ]:
# code here
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(model_h_clus.labels_, model_clus.labels_)

print(f"Adjusted Rand Index: {ari:.3f}") #Permite comparar dos clustering aunque los numero sean diferentes

In [ ]:
comparison = pd.DataFrame({
    'country': df['country'],
    'Hierarchical': model_h_clus.labels_,
    'KMeans': model_clus.labels_
})

print(comparison.sort_values('KMeans'))

### Application: clustering images

Theses techniques can be use to reduce the number of colours in a picture

- find a picture you want to reduce the number of colours.
- use `skimages` and `sklearn` to reduce the number of colours in this picture
- from your picture create two pictures with 16 colours what do you notice?

In [ ]:
# install skimage
import numpy as np
import matplotlib.pyplot as plt

from skimage import data
from skimage.color import rgb2lab

from sklearn.cluster import KMeans, AgglomerativeClustering
#Code here

In [ ]:
image = data.astronaut()

plt.imshow(image)
plt.axis('off')
plt.show()
print(image.shape)

512 rows of pixels, 512 columns of pixels and 3 RGB

In [ ]:
pixels = image.reshape(-1, 3) #Reshape the image in a matrix: each row is a pixel and 3 columns for RGB
#reshape cambia dimensiones de un array sin cambiar los datos: 3 (queremos 3 clumnas para RGB), -1 calcula los numeros de filas necesarias para que todo encaje
print(pixels.shape)

In [ ]:
kmeans = KMeans(n_clusters=16, random_state=42, n_init=10)

labels_kmeans = kmeans.fit_predict(pixels) #Primero fit para dividir la imagen en clusters y luego predict para
#calcular a que cluster pertenece cada pixel 

# Replace each pixel with its cluster centroid color
pixels_kmeans = kmeans.cluster_centers_[labels_kmeans]

# Reconstruct the image
image_kmeans = pixels_kmeans.reshape(image.shape).astype(np.uint8) #De float a enteros entre 
#0 y 255 que es lo que se suele usar para los pixeles

In [ ]:
from skimage.transform import resize

#reducimos la imagen porque calcular las parejas de 512x512 pixeles son millones, la reducimos a 64x64
small_image = resize(image, (64, 64), anti_aliasing=True) #Evitar artefactos visuales
small_image = (small_image * 255).astype(np.uint8)

pixels_small = small_image.reshape(-1, 3)

hierarchical = AgglomerativeClustering(n_clusters=16, linkage='ward')

labels_h = hierarchical.fit_predict(pixels_small)

# Calculate the mean colour of each cluster
pixels_h = np.zeros_like(pixels_small)

for cluster in range(16):
    mask = labels_h == cluster
    pixels_h[mask] = pixels_small[mask].mean(axis=0)

image_hierarchical = pixels_h.reshape(small_image.shape).astype(np.uint8)

In [ ]:
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.imshow(image)
plt.title('Original - 512x512')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(image_kmeans)
plt.title('KMeans - 16 colours')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(small_image)
plt.title('Original resized - 64x64')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(image_hierarchical)
plt.title('Hierarchical - 16 colours')
plt.axis('off')

plt.tight_layout()
plt.show()

Now let's try to apply KMeans clustering on the simple digits data that can be loaded from sklearn (https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

The goal is to use KMeans to try to identify similar digits without using the original label information.

Start by loading the digits and then finding the KMeans clusters. The digits data consists of 1,797 samples with 64 features, where each of the 64 features is the brightness of one pixel in an 8×8 image.

In [ ]:
from sklearn.datasets import load_digits
import pandas as pd
import matplotlib.pyplot as plt

digits = load_digits()

X = digits.data
y = digits.target

print(X.shape)

The cluster centroids themselves are 64-dimensional points, and can thus be interpreted as the "typical" digit within the cluster. Plot each cluster center. Hint: use *imshow*

In [ ]:
#Code here
kmeans = KMeans(n_clusters=10, random_state=42, n_init=100)

clusters = kmeans.fit_predict(X)

fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(kmeans.cluster_centers_[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Cluster {i}')
    ax.axis('off')

plt.tight_layout()
plt.show()

For each of the data points we also have access to their real labels (using *digits.target*). This means we can verify if the clustering algorithm split the data in such a way as to predict the original classes. 

Note that the labels of the discovered clusters may not directly match the initial labels in the dataset (they could be permuted). Write code to match each learned cluster label with the true labels of the data found in them.

In [ ]:
# code here
cluster_mapping = {}

for cluster in range(10):
    mask = clusters == cluster
    most_common = pd.Series(y[mask]).mode()[0]
    cluster_mapping[cluster] = most_common
y_pred = np.array([cluster_mapping[c] for c in clusters])


Now compute the accuracy score between the true labels from the data and the predicted ones.

In [ ]:
# code here
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, y_pred)
print(f"Accuracy: {accuracy:.2%}")

Plot the confusion matrix between the predicted labels and the actual labels. Which were the digits that were most often confused with other digits?

In [ ]:
# code here
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y, y_pred)

ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.show()

What if instead of keeping the original 64 features, we perform a projection into 2 dimensions?
Use the t-distributed stochastic neighbor embedding (t-SNE) algorithm to pre-process the data before performing KMeans. t-SNE is a nonlinear embedding algorithm that is particularly adept at preserving points within clusters.

Compute the new accuracy.

In [ ]:
# code here